In [28]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

In [29]:
df = pd.read_csv('/honeywell.csv')

# PREPROCESSING

In [30]:
# 1. Handle Missing Values
# Checking for nulls (though this dataset usually has none)
print("Missing values per column:\n", df.isnull().sum())
# If there were any, we could drop them or fill them:
df = df.dropna()

# 2. Remove or Cap Outliers using IQR (Interquartile Range)
# We apply this only to the sensor features
sensor_cols = ['Air temperature [K]', 'Process temperature [K]',
               'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

for col in sensor_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Capping outliers to the bounds (better than removing to keep 10k rows)
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

print("\nOutliers capped using IQR method.")

# 3. Encode Categorical Variables
# 'Type' is ordinal (L < M < H), but LabelEncoder or Mapping works well here
le = LabelEncoder()
df['Type'] = le.fit_transform(df['Type'])
# L=1, M=2, H=0 (alphabetical order) or custom map: df['Type'].map({'L': 0, 'M': 1, 'H': 2})

# 4. Normalize Numerical Features using StandardScaler
scaler = StandardScaler()
# Note: We don't scale the Target columns (Machine failure, TWF, etc.)
df[sensor_cols] = scaler.fit_transform(df[sensor_cols])

# Final touch: Drop identifiers that offer no predictive value
df_processed = df.drop(columns=['UDI', 'Product ID'])

print("\n--- Preprocessing Complete ---")
display(df_processed.head())

Missing values per column:
 UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Outliers capped using IQR method.

--- Preprocessing Complete ---


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,2,-0.952389,-0.947360,0.140180,0.284091,-1.695984,0,0,0,0,0,0
1,1,-0.902393,-0.879959,-0.820899,0.637122,-1.648852,0,0,0,0,0,0
2,1,-0.952389,-1.014761,-0.216024,0.949807,-1.617430,0,0,0,0,0,0
3,1,-0.902393,-0.947360,-0.652879,-0.048768,-1.586009,0,0,0,0,0,0
4,1,-0.902393,-0.879959,-0.820899,0.001665,-1.554588,0,0,0,0,0,0


# FEATURE ENGINEERING
- Temperature difference (process - air)

- Stress index (torque / rotational speed)

- torque_wear = torque * tool_wear -rolling_torque_std

- torque_change = torque - torque_lag1

- speed_change = speed - speed_lag1 -Lag features (previous 1 and 2 values)

In [31]:
# First, let's create cleaner names for calculation ease
t_air = 'Air temperature [K]'
t_process = 'Process temperature [K]'
torque = 'Torque [Nm]'
speed = 'Rotational speed [rpm]'
tool_wear = 'Tool wear [min]'

# 1. Temperature difference (process - air)
df['temp_diff'] = df[t_process] - df[t_air]

# 2. Stress index (torque / rotational speed)
# We add a small epsilon to speed to avoid division by zero
df['stress_index'] = df[torque] / (df[speed] + 1e-5)

# 3. Torque-Wear interaction
df['torque_wear'] = df[torque] * df[tool_wear]

# 4. Rolling Torque Standard Deviation
# Using a window of 5 cycles to see how much the torque is fluctuating
df['rolling_torque_std'] = df[torque].rolling(window=5).std().fillna(0)

# 5. Lag Features (Previous 1 and 2 values for torque and speed)
# This helps the model see "sudden changes" over time
df['torque_lag1'] = df[torque].shift(1).fillna(df[torque].median())
df['torque_lag2'] = df[torque].shift(2).fillna(df[torque].median())
df['speed_lag1'] = df[speed].shift(1).fillna(df[speed].median())
df['speed_lag2'] = df[speed].shift(2).fillna(df[speed].median())

# 6. Change Features (Current - Previous)
df['torque_change'] = df[torque] - df['torque_lag1']
df['speed_change'] = df[speed] - df['speed_lag1']

print("--- Feature Engineering Complete ---")
print(f"New Dataframe Shape: {df.shape}")
display(df[['temp_diff', 'stress_index', 'torque_wear', 'rolling_torque_std', 'torque_change']].head())

--- Feature Engineering Complete ---
New Dataframe Shape: (10000, 24)


,temp_diff,stress_index,torque_wear,rolling_torque_std,torque_change
0,0.005030,2.026471,-0.481813,0.000000,0.272339
1,0.022434,-0.776136,-1.050520,0.000000,0.353032
2,-0.062371,-4.396963,-1.536247,0.000000,0.312685
3,-0.044966,0.074698,0.077346,0.000000,-0.998575
4,0.022434,-0.002029,-0.002589,0.425973,0.050433


# Creating Time Sequences

In [32]:
def create_lstm_sequences(df, target_col, seq_length=10):
    """
    Converts a pandas DataFrame into 3D sequences for LSTM training.

    Parameters:
    df (pd.DataFrame): The preprocessed dataframe.
    target_col (str): The name of the label column (e.g., 'Machine failure').
    seq_length (int): Number of previous time steps to include in each sample.

    Returns:
    X (np.array): 3D array of shape (samples, seq_length, features)
    y (np.array): 1D array of labels
    """
    X = []
    y = []

    # Extract features (all columns except the target)
    features = df.drop(columns=[target_col]).values
    labels = df[target_col].values

    # Loop through the data to create overlapping sequences
    for i in range(len(df) - seq_length):
        # Grab a slice of 'seq_length' rows for features
        X.append(features[i : i + seq_length])
        # The label is the value at the very next time step
        y.append(labels[i + seq_length])

    return np.array(X), np.array(y)

# --- Usage Example ---
# sequence_length = 10
# X, y = create_lstm_sequences(df_processed, target_col='Machine failure', seq_length=10)

# print(f"Original Dataframe Shape: {df_processed.shape}")
# print(f"LSTM Input Shape (X): {X.shape}") # (9990, 10, num_features)
# print(f"LSTM Label Shape (y): {y.shape}") # (9990,)

# Feature Fusion

In [33]:
def create_final_feature_matrix(df, n_components=None):
    """
    Combines engineered features and optionally applies PCA.

    Parameters:
    df (pd.DataFrame): Dataframe containing all engineered features.
    n_components (int/float): If float (0-1), retains that % of variance.
                               If int, reduces to that many features.
                               If None, PCA is skipped.
    """

    # 1. Define the Feature Set
    # We exclude the identifiers and all failure labels
    targets = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
    identifiers = ['UDI', 'Product ID']

    # Select only columns that are NOT targets or identifiers
    feature_cols = [col for col in df.columns if col not in (targets + identifiers)]
    X = df[feature_cols].copy()

    print(f"Initial Feature Count: {X.shape[1]}")

    # 2. Re-scaling (Required for PCA)
    # Even if previously scaled, new engineered features (like torque_wear) need scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 3. Optional PCA Application
    if n_components is not None:
        pca = PCA(n_components=n_components)
        X_final = pca.fit_transform(X_scaled)

        explained_var = sum(pca.explained_variance_ratio_) * 100
        print(f"PCA complete. Reduced to {X_final.shape[1]} components.")
        print(f"Total Variance Retained: {explained_var:.2f}%")

        # Convert back to DataFrame for easier handling
        X_final = pd.DataFrame(
            X_final,
            columns=[f'PC{i+1}' for i in range(X_final.shape[1])]
        )
    else:
        # If no PCA, return the scaled engineered feature matrix
        X_final = pd.DataFrame(X_scaled, columns=feature_cols)
        print("Skipping PCA. Returning full engineered feature matrix.")

    return X_final, df['Machine failure']

# --- Execution ---
# Option A: Keep all features
X_full, y = create_final_feature_matrix(df)

# Option B: Apply PCA to retain 95% of the information
X_pca, y = create_final_feature_matrix(df, n_components=0.95)

Initial Feature Count: 16
Skipping PCA. Returning full engineered feature matrix.
Initial Feature Count: 16
PCA complete. Reduced to 10 components.
Total Variance Retained: 97.87%


# train test split

In [34]:
def time_series_split(X, y, train_ratio=0.7, val_ratio=0.15):
    """
    Splits data into Train, Validation, and Test sets chronologically.

    Parameters:
    X, y: Features and Labels (can be DataFrames or Numpy arrays)
    train_ratio: Proportion for training (default 70%)
    val_ratio: Proportion for validation (default 15%)
    """
    n = len(X)

    # Calculate split indices
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    # Slice the data
    X_train, y_train = X[:train_end], y[:train_end]
    X_val, y_val = X[train_end:val_end], y[train_end:val_end]
    X_test, y_test = X[val_end:], y[val_end:]

    print(f"Total samples: {n}")
    print(f"Train set:      {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")
    print(f"Test set:       {X_test.shape[0]} samples")

    return X_train, X_val, X_test, y_train, y_val, y_test

# --- Execution ---
# Using the X and y derived from your feature engineering/scaling steps
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(X_pca, y)

Total samples: 10000
Train set:      7000 samples
Validation set: 1500 samples
Test set:       1500 samples


"My decision to perform feature engineering and sequence generation prior to the split was a deliberate architectural choice to preserve the temporal continuity and physical integrity of the machine's operational signal. In a time-series context, features such as rolling standard deviations and multi-step lags rely on a continuous historical stream; splitting the raw data first would create 'cold-start' boundary errors at the beginning of the validation and test sets, depriving the LSTM of the necessary context to make accurate predictions. Furthermore, because I employed a strict chronological split rather than a random shuffle, I have effectively prevented temporal data leakage, as the model remains entirely blind to the target labels of the future test period, ensuring that the engineered features represent independent physical states rather than statistical leaks from the future."

In [35]:
import pandas as pd

def save_processed_data(X, y, filename='processed_maintenance_data.csv'):
    """
    Combines features and labels into one DataFrame and saves to CSV.
    """
    # 1. Ensure X is a DataFrame (if it was a numpy array from Scaling/PCA)
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])

    # 2. Reset indices to ensure they align perfectly before concatenation
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    # 3. Concatenate features and the target label
    final_df = pd.concat([X, y], axis=1)

    # 4. Save to CSV
    final_df.to_csv(filename, index=False)
    print(f"Successfully saved {final_df.shape[0]} rows to {filename}")
    np.save('X_sequences.npy', X)  # The 3D array for LSTM
    np.save('y_labels.npy', y)

print("Sequences saved as .npy files for LSTM training.")

# --- Execution ---
# Save your full engineered feature set
save_processed_data(X_full, y, 'ai4i_engineered_features.csv')

# If you want to download it from Google Colab to your local machine:
from google.colab import files
files.download('ai4i_engineered_features.csv')

Sequences saved as .npy files for LSTM training.
Successfully saved 10000 rows to ai4i_engineered_features.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>